In [ ]:
from pathlib import Path
import re
import socket

import pandas as pd
import pyfame as pf
from pyfame.pyfame import BoEFAMEReader

# -----------------------------
# FAME extraction configuration
# -----------------------------
VINTAGE_START = "M15"
VINTAGE_END = "M26"
FREQUENCY = "quarterly"
END_OF_PERIOD_DATES = True
EXPORT_TO_CSV_FROM_PYFAME = False

# pyfame currently calls this local SOAP endpoint internally.
FAME_LOCAL_HOST = "localhost"
FAME_LOCAL_PORT = 12345

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "data" / "fame_exports"
RAW_DIR = OUTPUT_DIR / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

# CPI disaggregation inputs used by fcst_cpi_components.R
# These codes/expressions follow the Estimation_data sheet logic in data_set_v3.xlsx.
ESTIMATION_SERIES_MAP = {
    "date": "date",
    "cpi": "stifidx",
    "core_gds": "stif_gds_xeatfnab",
    "services": "stif_srv",
    "food_at": "food_at_expr",
    "energy": "stif_pu",
    "pmdef": "pmdef",
    "eer": "eer",
    "infl_exp": "infl_exp_expr",
}

BASELINE_SERIES_MAP = {
    "date": "date",
    # Index series share FAME codes with the estimation panel (conv2qa(X12(...))).
    "stif_cpi": "stifidx",
    "core_gds": "stif_gds_xeatfnab",
    "services": "stif_srv",
    "food_at":  "food_at_expr",
    "energy_f": "stif_pu",
    "pmdef_f":  "pmdef",
    "eer_f":    "eer",
    # Forecast / baseline-specific series.
    "infl_exp_f": "infl_exp_expr",   # same BPROJA expression as estimation infl_exp
    "cpi_f":      "cpi_f_expr",      # FAME: cpisa
    "encont_f":   "encont_f_expr",   # FAME: M26_encont_fin.q
    # Weights sourced directly from FAME via CONV2QA(*_wgt.m).
    "s_wgt": "stif_srv_wgt",
    "e_wgt": "stif_pu_wgt",
    "cg_wgt": "STIF_GDS_XEATFNAB_wgt",
    # f_wgt = food + alcohol + tobacco weights (synthesized after build_panel).
    "_stif_fnab_wgt": "stif_fnab_wgt",
    "_booze_wgt": "booze_wgt",
    "_fags_wgt": "fags_wgt",
}

# Expression-based pulls for series that are transformed in Excel sheets.
INFL_EXP_FAME_EXPR = "BPROJA(conv2qa(cgrp.m),PUB12MEDIAN.Q,\"E\")"

EXPRESSION_OVERRIDES = {
    "stifidx": "conv2qa(X12(stifidx.m,MULT,1990m2,2025m12))",
    "stif_srv": "conv2qa(X12(stif_srv.m,MULT,1990m2,2025m12))",
    "stif_gds_xeatfnab": "conv2qa(X12(stif_gds_xeatfnab.m,MULT,1990m2,2025m12))",
    "food_at_expr": "conv2qa(X12(cpi_aggregation(stif_fnab.m, booze.m, fags.m,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#),MULT,1990m2,2025m12))",
    "stif_pu": "conv2qa(X12(stif_pu.m,MULT,1990m2,2025m12))",
    "infl_exp_expr": INFL_EXP_FAME_EXPR,
    "STIF_GDS_XEATFNAB_wgt": "CONV2QA(STIF_GDS_XEATFNAB_wgt.m)",
    "cpi_f_expr":    "cpisa",
    "encont_f_expr": "M26_encont_fin.q",
    "stif_srv_wgt":  "CONV2QA(stif_srv_wgt.m)",
    "stif_pu_wgt":   "CONV2QA(stif_pu_wgt.m)",
    "stif_fnab_wgt": "CONV2QA(stif_fnab_wgt.m)",
    "booze_wgt":     "CONV2QA(booze_wgt.m)",
    "fags_wgt":      "CONV2QA(fags_wgt.m)",
}

REQUIRED_ESTIMATION_COLUMNS = [
    "date", "cpi", "core_gds", "services", "food_at", "energy", "pmdef", "eer", "infl_exp"
]

REQUIRED_BASELINE_COLUMNS = [
    "date", "stif_cpi", "core_gds", "services", "food_at", "energy_f", "pmdef_f",
    "eer_f", "infl_exp_f", "cpi_f", "s_wgt", "f_wgt", "e_wgt", "cg_wgt", "encont_f"
]

ALL_CODES = sorted(
    set(v for k, v in ESTIMATION_SERIES_MAP.items() if k != "date")
    | set(v for k, v in BASELINE_SERIES_MAP.items() if k != "date")
)


_VINTAGE_LETTERS = ["F", "M", "A", "N"]  # Q1, Q2, Q3, Q4


def _round_to_index(label: str) -> int:
    m = re.match(r"^([FMAN])(\d{2})$", str(label).strip().upper())
    if not m:
        raise ValueError(f"Bad round label: {label!r}")
    yy = int(m.group(2))
    return yy * 4 + _VINTAGE_LETTERS.index(m.group(1))


def normalize_quarter_end_dates(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    """Standardize quarterly timestamps to quarter-end, including quarter-start labels."""
    out = df.copy()
    raw = out[date_col]
    numeric = pd.to_numeric(raw, errors="coerce")
    if numeric.notna().any() and pd.api.types.is_numeric_dtype(raw):
        dt = pd.to_datetime(numeric, unit="ms", errors="coerce")
    else:
        dt = pd.to_datetime(raw, errors="coerce")
        # Guard against large numeric strings interpreted as nanoseconds.
        if dt.notna().any() and dt.dropna().dt.year.median() < 1980 and numeric.notna().any():
            dt_ms = pd.to_datetime(numeric, unit="ms", errors="coerce")
            if dt_ms.notna().sum() >= dt.notna().sum():
                dt = dt_ms
    quarter_start = dt.dt.is_quarter_start.fillna(False)
    dt = dt.where(~quarter_start, dt - pd.Timedelta(days=1))
    out[date_col] = dt.dt.to_period("Q").dt.to_timestamp(how="end").dt.normalize()
    return out


def _vintage_sort_key(name: str):
    text = str(name).strip().upper()
    m = re.fullmatch(r"([FMAN])(\d{2})", text)
    if m:
        return (1, _round_to_index(text))
    match = re.search(r"(\d+)", text)
    return (0, int(match.group(1))) if match else (-1, -1)


def latest_vintage_column(df: pd.DataFrame) -> str:
    candidates = [c for c in df.columns if c != "date"]
    if not candidates:
        raise ValueError("No vintage columns found. Expected at least one column besides 'date'.")
    return sorted(candidates, key=_vintage_sort_key)[-1]


def latest_series(df: pd.DataFrame, output_name: str) -> pd.DataFrame:
    col = latest_vintage_column(df)
    return df[["date", col]].rename(columns={col: output_name}).copy()


def build_panel(raw_data: dict, mapping: dict) -> pd.DataFrame:
    panel = None
    for output_name, fame_code in mapping.items():
        if output_name == "date":
            continue
        if fame_code not in raw_data:
            raise KeyError(f"Missing FAME code in downloaded data: {fame_code}")
        current = latest_series(raw_data[fame_code], output_name)
        panel = current if panel is None else panel.merge(current, on="date", how="outer")
    return panel.sort_values("date").reset_index(drop=True)


def reorder_and_validate(df: pd.DataFrame, required_columns: list, dataset_name: str) -> pd.DataFrame:
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"{dataset_name} missing columns: {missing}")
    return df[required_columns].sort_values("date").reset_index(drop=True)


def is_port_open(host: str, port: int, timeout: float = 2.0) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(timeout)
        return s.connect_ex((host, port)) == 0


def ensure_fame_service_available(host: str = FAME_LOCAL_HOST, port: int = FAME_LOCAL_PORT):
    if not is_port_open(host, port):
        raise ConnectionError(
            f"FAME local service is not reachable at {host}:{port}. "
            "Please start the local FAME web service/tunnel and retry Cell 3."
        )


print("Setup complete for CPI disaggregation data extraction.")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Total FAME codes requested: {len(ALL_CODES)}")
print(f"FAME endpoint expected by pyfame: {FAME_LOCAL_HOST}:{FAME_LOCAL_PORT}")

### Download CPI-disaggregation inputs from FAME

In [10]:
ensure_fame_service_available()

# Request all CPI disaggregation series from FAME (estimation + baseline).
requested_codes = ALL_CODES

raw_data = {}
reader = BoEFAMEReader()

for code in requested_codes:
    # Expression-based path mirrors Excel transformations where needed.
    if code in EXPRESSION_OVERRIDES:
        converted = reader.read(EXPRESSION_OVERRIDES[code])
        if converted is None or converted.empty:
            raise ValueError(f"FAME returned empty data for series: {code}")

        converted = converted.rename(columns={"index": "date", "value": VINTAGE_END})
        converted["date"] = pd.to_datetime(converted["date"], unit="ms", errors="coerce")
        converted = normalize_quarter_end_dates(converted)
        converted = converted.dropna(subset=["date", VINTAGE_END]).sort_values("date").reset_index(drop=True)

        raw_data[code] = converted
        file_path = RAW_DIR / f"{code}_vintages.parquet"
        converted.to_parquet(file_path, index=False)
        continue

    try:
        result = pf.download_vintaged_fame_data(
            variables=[code],
            vintage_start=VINTAGE_START,
            vintage_end=VINTAGE_END,
            frequency=FREQUENCY,
            end_of_period_dates=END_OF_PERIOD_DATES,
            export_to_csv=EXPORT_TO_CSV_FROM_PYFAME,
        )
    except ValueError as exc:
        if "All objects passed were None" in str(exc):
            raise RuntimeError(f"No data returned from FAME for series: {code}") from exc
        raise

    if not isinstance(result, dict) or code not in result:
        raise KeyError(f"Missing data series from FAME response: {code}")

    df = result[code]
    if df is None or df.empty:
        raise ValueError(f"FAME returned empty data for series: {code}")
    df = normalize_quarter_end_dates(df)

    raw_data[code] = df
    file_path = RAW_DIR / f"{code}_vintages.parquet"
    df.to_parquet(file_path, index=False)

print("Data source used: FAME")
print(f"Downloaded and saved raw vintages for {len(requested_codes)} series: {requested_codes}")
print(f"Raw files folder: {RAW_DIR}")

100%|██████████| 45/45 [00:34<00:00,  1.32it/s]


Data source used: FAME
Downloaded and saved raw vintages for 21 series: ['STIF_GDS_XEATFNAB_wgt', 'cpi_f', 'e_wgt', 'eer', 'eer_f', 'encont_f', 'energy_f', 'f_wgt', 'food_at', 'food_at_expr', 'infl_exp_expr', 'infl_exp_f', 'pmdef', 'pmdef_f', 's_wgt', 'services', 'stif_cpi', 'stif_gds_xeatfnab', 'stif_pu', 'stif_srv', 'stifidx']
Raw files folder: c:\Users\344792\Gokce\GIT PROJECTS\DisaggCPI\CPI-disaggregation-in-PT\data\fame_exports\raw


### Pull full vintage panel for inflation variables via `{ROUND}_*_FIN.m`

For the 5 inflation variables consumed by the forecast-evaluation dashboard, FAME stores per-round snapshots under `{ROUND}_{code}_FIN.m` (already seasonally-adjusted and aggregated to quarterly when read via `BoEFAMEReader`). This cell loops over `F15…M26`, evaluates the per-vintage expression for each variable, and overwrites the matching raw `*_vintages.parquet` files so the downstream outturn-build cell picks up the multi-vintage panel automatically.

In [3]:
def _index_to_round(idx: int) -> str:
    return f"{_VINTAGE_LETTERS[idx % 4]}{idx // 4:02d}"


def enumerate_vintage_rounds(start: str, end: str) -> list:
    return [_index_to_round(i) for i in range(_round_to_index(start), _round_to_index(end) + 1)]


# Per-vintage expression templates for the 5 inflation variables.
# Primary form is the published quarterly SA value (`{ROUND}_*_FIN.m`).
# Fallback wraps the benchmark monthly (`{ROUND}_*_bmk.m`) in the workbook's
# conv2qa(X12(...)) transformation, used when FIN was not archived for a round.
INFLATION_VINTAGE_EXPRS = {
    "stifidx": {
        "primary":  "{ROUND}_stifidx_FIN.q",
        "fallback": "conv2qa({ROUND}_stifidx_bmk.m)",
    },
    "stif_srv": {
        "primary":  "{ROUND}_stif_srv_FIN.q",
        "fallback": "conv2qa({ROUND}_stif_srv_bmk.m)",
    },
    "stif_gds_xeatfnab": {
        "primary":  "{ROUND}_stif_gds_xeatfnab_FIN.q",
        "fallback": "conv2qa({ROUND}_stif_gds_xeatfnab_bmk.m)",
    },
    "stif_pu": {
        "primary":  "{ROUND}_stif_pu_FIN.q",
        "fallback": "conv2qa({ROUND}_stif_pu_bmk.m)",
    },
    "food_at_expr": {
        "primary":  "conv2qa(X12(cpi_aggregation({ROUND}_stif_fnab_FIN.q, booze.m, fags.m,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#),MULT,1996m1,2029m12))",
        "fallback": "conv2qa(X12(cpi_aggregation({ROUND}_stif_fnab_bmk.m, booze.m, fags.m,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#,#),MULT,1996m1,2029m12))",
    },
}


def _try_read(expr: str):
    """Read an expression; return DataFrame or None on empty/failure."""
    try:
        df = reader.read(expr)
    except Exception:
        return None
    if df is None or df.empty:
        return None
    df = df.rename(columns={"index": "date"}).copy()
    df["date"] = pd.to_datetime(df["date"], unit="ms", errors="coerce")
    df = normalize_quarter_end_dates(df)
    return df


VINTAGE_ROUNDS = enumerate_vintage_rounds("F15", VINTAGE_END)
print(f"Pulling {len(VINTAGE_ROUNDS)} vintages ({VINTAGE_ROUNDS[0]}..{VINTAGE_ROUNDS[-1]}) "
      f"for {len(INFLATION_VINTAGE_EXPRS)} inflation variables.")

vintage_results = {}
for code, templates in INFLATION_VINTAGE_EXPRS.items():
    panel = None
    available_rounds = []
    fallback_rounds = []
    for rnd in VINTAGE_ROUNDS:
        df = _try_read(templates["primary"].replace("{ROUND}", rnd))
        source = "FIN"
        if df is None:
            df = _try_read(templates["fallback"].replace("{ROUND}", rnd))
            source = "bmk"
        if df is None:
            continue
        s = df.rename(columns={"value": rnd}).copy()
        s = normalize_quarter_end_dates(s)
        s = s.dropna(subset=["date", rnd])
        if s.empty:
            continue
        available_rounds.append(rnd)
        if source == "bmk":
            fallback_rounds.append(rnd)
        panel = s if panel is None else panel.merge(s, on="date", how="outer")

    if panel is None or panel.empty:
        raise RuntimeError(f"No vintage data retrieved for {code}")

    panel = panel.sort_values("date").reset_index(drop=True)
    out = RAW_DIR / f"{code}_vintages.parquet"
    panel.to_parquet(out, index=False)
    vintage_results[code] = (len(available_rounds), available_rounds[0], available_rounds[-1], fallback_rounds)
    raw_data[code] = panel
    note = f" (bmk fallback for {fallback_rounds})" if fallback_rounds else ""
    print(f"  {code}: {len(available_rounds)} vintages "
          f"({available_rounds[0]}..{available_rounds[-1]}) -> {out.name}{note}")

print("\nMulti-vintage refresh complete.")


Pulling 46 vintages (F15..M26) for 5 inflation variables.
  stifidx: 46 vintages (F15..M26) -> stifidx_vintages.parquet
  stif_srv: 46 vintages (F15..M26) -> stif_srv_vintages.parquet
  stif_gds_xeatfnab: 46 vintages (F15..M26) -> stif_gds_xeatfnab_vintages.parquet
  stif_pu: 46 vintages (F15..M26) -> stif_pu_vintages.parquet
  food_at_expr: 46 vintages (F15..M26) -> food_at_expr_vintages.parquet

Multi-vintage refresh complete.


### Build estimation dataset (matches `Estimation_data` in `data_set_v3`)

In [11]:
estimation_data = build_panel(raw_data, ESTIMATION_SERIES_MAP)
estimation_data = estimation_data.dropna(
    how="all", subset=[c for c in estimation_data.columns if c != "date"]
)
estimation_data = reorder_and_validate(
    estimation_data, REQUIRED_ESTIMATION_COLUMNS, "Estimation_data"
)

print("Built estimation dataset.")
print(estimation_data.tail(3))

Built estimation dataset.
          date       cpi  core_gds  services    food_at    energy     pmdef  \
169 2028-12-31  1.001070  1.001070  1.001070  82.530538  1.001070  1.002358   
170 2029-03-31  0.999799  0.999799  0.999799  82.415652  0.999799  1.001070   
171 2029-06-30       NaN       NaN       NaN        NaN       NaN  0.999799   

           eer  infl_exp  
169  82.648533       NaN  
170  82.530538       NaN  
171  82.415652       NaN  


### Build baseline forecast dataset (matches `M26_Baseline` in `data_set_v3`)

In [ ]:
baseline_data = build_panel(raw_data, BASELINE_SERIES_MAP)
# Synthesize f_wgt = food + alcohol + tobacco weights (matches Excel formula).
_fwgt_parts = ["_stif_fnab_wgt", "_booze_wgt", "_fags_wgt"]
baseline_data["f_wgt"] = baseline_data[_fwgt_parts].sum(axis=1, min_count=1)
baseline_data = baseline_data.drop(columns=_fwgt_parts)
baseline_data = baseline_data.dropna(
    how="all", subset=[c for c in baseline_data.columns if c != "date"]
)
baseline_data = reorder_and_validate(
    baseline_data, REQUIRED_BASELINE_COLUMNS, "M26_Baseline"
)

print("Built baseline dataset.")
print(baseline_data.tail(3))

### Finalize units and export forecast-ready files

In [6]:
# Keep raw units from FAME; fcst_cpi_components.R handles any scaling needed.

# Save direct-use files for Python/R pipelines
estimation_data.to_parquet(OUTPUT_DIR / "estimation_data_latest.parquet", index=False)
estimation_data.to_csv(OUTPUT_DIR / "estimation_data_latest.csv", index=False)

baseline_data.to_parquet(OUTPUT_DIR / "baseline_data_latest.parquet", index=False)
baseline_data.to_csv(OUTPUT_DIR / "baseline_data_latest.csv", index=False)

print("Exports updated: estimation and baseline datasets.")
print(f"Estimation date range: {estimation_data['date'].min()} -> {estimation_data['date'].max()}")
print(f"Baseline date range: {baseline_data['date'].min()} -> {baseline_data['date'].max()}")

Exports updated: estimation and baseline datasets.
Estimation date range: 1986-09-30 00:00:00 -> 2029-06-30 00:00:00
Baseline date range: 1986-12-31 00:00:00 -> 2031-12-31 00:00:00


### Build outturn parquets for forecast-evaluation dashboard

One file per inflation variable (`outturn_data{cpi,s,cg,f,e}.parquet`), long format `date, vintage_date, value, frequency, forecast_horizon, variable`. Consumed by `New-script.py` via `forecast_evaluation`.

Currently only the M26 vintage is available for the `stif*` derived series. When the vintaged FAME database-context syntax is resolved, the raw `{code}_vintages.parquet` files will gain additional vintage columns and this cell will pick them up unchanged.

In [7]:
VINTAGE_LETTER_TO_Q = {"F": 1, "M": 2, "A": 3, "N": 4}


def vintage_label_to_quarter(label: str) -> str:
    """Convert FAME round label (e.g. 'M26') to a 'YYYY Q#' string."""
    m = re.match(r"^([FMAN])(\d{2})$", str(label).strip().upper())
    if not m:
        raise ValueError(f"Unrecognised vintage label: {label!r}")
    letter, yy = m.group(1), m.group(2)
    return f"{2000 + int(yy)} Q{VINTAGE_LETTER_TO_Q[letter]}"


def date_to_quarter(ts) -> str:
    """Convert a Timestamp/date to a 'YYYY Q#' string (quarter that contains it)."""
    p = pd.Period(pd.Timestamp(ts), freq="Q")
    return f"{p.year} Q{p.quarter}"


# Output filename suffix -> (FAME code stored in RAW_DIR, human-readable label)
OUTTURN_VARIABLES = {
    "cpi": ("stifidx",            "CPI inflation"),
    "s":   ("stif_srv",           "Services inflation"),
    "cg":  ("stif_gds_xeatfnab",  "Core goods inflation"),
    "f":   ("food_at_expr",       "Food, alcohol & tobacco inflation"),
    "e":   ("stif_pu",            "Energy inflation"),
}

OUTTURN_DIR = PROJECT_ROOT / "data"
OUTTURN_DIR.mkdir(parents=True, exist_ok=True)

outturn_summary = []
for suffix, (code, var_label) in OUTTURN_VARIABLES.items():
    raw_path = RAW_DIR / f"{code}_vintages.parquet"
    if not raw_path.exists():
        raise FileNotFoundError(f"Missing raw vintage file for {code}: {raw_path}")

    raw_df = pd.read_parquet(raw_path)
    vintage_cols = [c for c in raw_df.columns if c != "date"]
    if not vintage_cols:
        raise ValueError(f"{code}: no vintage columns to melt")

    long = raw_df.melt(
        id_vars=["date"],
        value_vars=vintage_cols,
        var_name="vintage_label",
        value_name="value",
    ).dropna(subset=["value"])

    long["vintage_date"] = long["vintage_label"].map(vintage_label_to_quarter)
    long["date"] = long["date"].map(date_to_quarter)
    long["variable"] = var_label
    long["frequency"] = "Q"

    d_periods = pd.PeriodIndex(long["date"].str.replace(r"\s+", "", regex=True), freq="Q")
    v_periods = pd.PeriodIndex(long["vintage_date"].str.replace(r"\s+", "", regex=True), freq="Q")
    long["forecast_horizon"] = (d_periods - v_periods).map(lambda x: x.n).astype(float)

    long = long[["date", "vintage_date", "value", "frequency", "forecast_horizon", "variable"]]
    long = long.sort_values(["vintage_date", "date"]).reset_index(drop=True)

    out_path = OUTTURN_DIR / f"outturn_data{suffix}.parquet"
    long.to_parquet(out_path, index=False)
    outturn_summary.append((out_path.name, len(long), long["vintage_date"].nunique()))

print("Outturn parquets written:")
for name, n_rows, n_vint in outturn_summary:
    print(f" - {name}: {n_rows} rows across {n_vint} vintage(s)")


Outturn parquets written:
 - outturn_datacpi.parquet: 7774 rows across 46 vintage(s)
 - outturn_datas.parquet: 7774 rows across 46 vintage(s)
 - outturn_datacg.parquet: 7774 rows across 46 vintage(s)
 - outturn_dataf.parquet: 7774 rows across 46 vintage(s)
 - outturn_datae.parquet: 7774 rows across 46 vintage(s)


In [8]:
output_files = [
    OUTPUT_DIR / "estimation_data_latest.parquet",
    OUTPUT_DIR / "estimation_data_latest.csv",
    OUTPUT_DIR / "baseline_data_latest.parquet",
    OUTPUT_DIR / "baseline_data_latest.csv",
]
output_files += [OUTTURN_DIR / f"outturn_data{s}.parquet" for s in OUTTURN_VARIABLES]

print("Export summary:")
for path in output_files:
    print(f" - {path.name}: {'OK' if path.exists() else 'MISSING'}")

print("\nColumns check:")
print("Estimation:", estimation_data.columns.tolist())
print("Baseline:", baseline_data.columns.tolist())


Export summary:
 - estimation_data_latest.parquet: OK
 - estimation_data_latest.csv: OK
 - baseline_data_latest.parquet: OK
 - baseline_data_latest.csv: OK
 - outturn_datacpi.parquet: OK
 - outturn_datas.parquet: OK
 - outturn_datacg.parquet: OK
 - outturn_dataf.parquet: OK
 - outturn_datae.parquet: OK

Columns check:
Estimation: ['date', 'cpi', 'core_gds', 'services', 'food_at', 'energy', 'pmdef', 'eer', 'infl_exp']
Baseline: ['date', 'stif_cpi', 'core_gds', 'services', 'food_at', 'energy_f', 'pmdef_f', 'eer_f', 'infl_exp_f', 'cpi_f', 's_wgt', 'f_wgt', 'e_wgt', 'cg_wgt', 'encont_f']
